# Introduction to STRAUSS (V2)

In this notebook, we will take a look at some of the ways you can use STRAUSS to turn data into sound (sonification).

## How to Use

Start by running the cell directly below this one (click the play button next to it). This will fetch all the required assets and install the correct branch of STRAUSS. **This can take a couple of minutes, so I would recommend doing this now!**

Once complete, you can run each snippet of code below to see/hear the results.

In [1]:
import os
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

REPO_URL = "https://github.com/gcaselton/creative-code-club.git"
REPO_NAME = "creative-code-club"
BRANCH = "main"

if IN_COLAB:
    # Only clone if we aren't already inside the cloned repo
    if Path.cwd().name != REPO_NAME:
        if not Path(REPO_NAME).exists():
            print("Cloning!")
            !git clone -q --depth 1 -b {BRANCH} {REPO_URL}
        os.chdir(REPO_NAME)
    
    # Install latest dev brach of STRAUSS
    %pip install -q git+https://github.com/james-trayford/strauss.git@note-aliases
    
DATA_DIR = Path('data')
STYLES_DIR = Path('styles')
STYLES_DIR.mkdir(parents=True, exist_ok=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Data

First, we will generate some artificial data using numpy and plot it.

In [2]:
!ls -la

total 13668
drwxr-xr-x 6 root root     4096 Nov 18 16:10 .
drwxr-xr-x 1 root root     4096 Nov 18 16:09 ..
drwxr-xr-x 4 root root     4096 Nov 18 16:09 bonus
drwxr-xr-x 8 root root     4096 Nov 18 16:09 .git
-rw-r--r-- 1 root root     4664 Nov 18 16:09 .gitignore
-rw-r--r-- 1 root root 13953867 Nov 18 16:09 intro.ipynb
-rw-r--r-- 1 root root       31 Nov 18 16:09 README.md
-rw-r--r-- 1 root root       62 Nov 18 16:09 requirements.txt
drwxr-xr-x 4 root root     4096 Nov 18 16:09 samples
drwxr-xr-x 2 root root     4096 Nov 18 16:36 styles


In [ ]:
import strauss
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 100, 1000) # 1000 equally spaced data points from 0-100
y = np.sin(x/4) # Sine curve

plt.xlabel('X')
plt.ylabel('Y')

plt.plot(x, y)
plt.show()


Next, we will use STRAUSS's new `sonify()` function to quickly get an audio representation of the data.

The default setting maps the `X` axis to time, and the `Y` axis to the pitch of a synthesised tone (triangle wave).

In [ ]:
strauss.sonify(x, y)
strauss.display() # Used to render the audio player in notebook environments
strauss.close() # we use this to prevent layering every sonification in the notebook - more on this later!

We can add keywords to the `sonify()` function, for example to change the duration of the sonification.

This will stretch (or compress) the data points over a different range of time.

In [ ]:
strauss.sonify(x, y, duration=4) # duration specified in seconds
strauss.display()
strauss.close()

Congratulations, you have just created a police siren!

---

# Styles

We can also use the `style` keyword to change the overall sound of the sonification.

In STRAUSS, a *Style* contains instructions on how to sonify the data.

The 3 key things a Style dictates are:

1. The underlying base sound (or instrument) used.
2. The parameter mappings, I.E. which data inputs are going to be mapped to which properties of the sound?
3. The musical notes used.

STRAUSS comes with some built-in Styles you can use, for example:

- `default`: This is the one you heard above! By omitting the `style` keyword, we use the default.
- `windy`: Maps the Y axis to a low-pass filter cutoff and uses a white noise base sound to create a textural and 'windy' effect.
- `mallets`: Maps the Y axis to pitches in a major scale, using individual mallet hits as the base sound.
- `flute_section`: Maps the Y axis to filter cutoff and uses 5 continuously-held flute notes to articulate a muscial chord.


---

Let's have a go at using one.

*The first time you run the below cell, it will need to download the flute samples.*

In [ ]:
strauss.sonify(x, y, style='flute_section')
strauss.display()
strauss.close()

Wowzers.

---

# Layering Sonifications

We can also layer different sonifications on top of each other to hear multiple data variables change simultaneously.

Under the hood, STRAUSS V2 uses the concept of an 'Audio Figure' (analagous to a figure in `matplotlib`) to act as the canvas upon which to add your sonifications.

Each time you call `strauss.sonify()`, it generates your sonification and adds it to the Audio Figure. This is why each of the cells above use `strauss.close()` to close the Audio Figure, ensuring that each sonification remains independent.

## Pro Tips!
- Currently, all sonifications added to the Audio Figure must have the same duration.
- Instead of using the 'module-level' Audio Figure by calling `strauss.sonify()`, you can create an Audio Figure object and use the same functions on it. For example:

```python
from strauss.audio_figure import AudioFigure

af = AudioFigure()
af.sonify(x, y)

```

This may be useful if you are using STRAUSS in an environment where you need a unique Audio Figure per use (in a web app, for example).

Let's hear an example by creating a new dataset, and sonifying it with a different style.

In [ ]:
z = -y

plt.xlabel('X')
plt.ylabel('Y/Z')

plt.plot(x, y)
plt.plot(x, z)
plt.show()

strauss.sonify(x, y, style='flute_section')
strauss.sonify(x, z, style='windy', level=0.7) # We can also use the 'level' keyword to set volume levels

strauss.display()
strauss.close()

# Creating Your Own Styles

*"Using presets is boring"*

I hear you! Let's take a look at how we can create custom Styles.

The information for each Style is saved in a Style File - which follows the yaml format. These are essentially key-value pairs, so we can build one in Python using a dictionary, and then save it as a .yml file.

Let's take a look at a basic Style to see what goes into one.

In [ ]:
import yaml

style = {
    
    "name": "My Basic Style", 
    
    "map":
        [
            {
                "output": "time"
            },
            {
                "output": "cutoff", 
                "output_range": [0, 0.8]        
            }
        ],
        
    "sources": "objects", 
    
    "notes": ["A2", "E3", "C#4", "B5"]
}

basic_style = str(STYLES_DIR / "basic.yml")

with open(basic_style, 'w') as file:
    yaml.dump(style, file)

strauss.sonify(x, y, style=basic_style)
strauss.display()
strauss.close()

There's a lot going on there!

Let's explain each element of the Style in detail.

## `name`

This is just used for metadata (can be anything you like), but it is required.

## `map`

The parameter mappings (I.E, which data variables will be controlling which sound properties).

You should list these in the order which you give the data arrays to the `sonify()` function. For example, in the Style above, we list the outputs as `time` first, then `cutoff`. That's because we want to map `x` to `time` and `y` to `cutoff`, so we call `sonify()` with `x` as the first argument, and `y` as the second.

Note that the `map` is given as a list of dictionaries.

### Mappings

So what goes in to each of those individual mapping dictionaries in the `map` list?

- `output`: The output sound parameter that the input data will control.
- `output_range` (optional): The lower and upper limits of the parameter (usually between 0 and 1).
- `input` (optional): You can use this to specify the column name if using a pandas DataFrame (more on this later).
- `input_range` (optional): The lower and upper bounds of the input data (if different to the min and max data points).
- `function` (optional): A mathematical function to apply when mapping the data.

## `sources`

This can be either `objects` or `events`, depending on how you want the source data to be played.

- `objects`: Sound is played continuously. Best suited to represent data that evolves continuously over time.
- `events`: Individual sounds are triggered over the duration of the sonification. Best suited to data where each data point is a discrete entity.

## `notes`

The musical notes used in the sonification. How these are played depends on whether you are using `objects` or `events`.

If using `objects`, all notes are held simultaneously (as a chord) for the duration of the sonification.
If using `events`, you can hear each individual note by mapping something to `pitch` as an `output`.

You can also give the name of a chord instead of listing individual notes, e.g. `"Am9"`!

---

*"I ain't reading all that!*

Fair enough! Back to the presets you go.

# Generator

There is one more important thing that goes into a Style: the `generator`. 

Earlier, I mentioned that the base sound/instrument was one of the key things that lives in a Style. The `generator` is where it is defined.

It was deliberately left out in the above example to keep things simple (it reverts to the default synth if no `generator` is given).

Let's now create a Style using the `sampler` generator type to use our own samples.

In [ ]:
style = {
    
    "name": "Harp",
    
    "generator": 
        {
            "type": "sampler",
            "sample": "./samples/Harp" # File path to the directory containing the samples
        },
    
    "map":
        [
            {
                "output": "time"
            },
            {
                "output": "pitch",      
            }
        ],
        
    "sources": "events",
    
    # This avoids hundreds of notes playing every second (remove at your own peril)
    "max_notes_per_sec": 10, 
    
    "notes": ["A2", "B2", "D3", "E3", "A3", "B3", "D4", "E4", "A5", "B4"]
}

harp_style = str(STYLES_DIR / "harp.yml")

with open(harp_style, 'w') as file:
    yaml.dump(style, file)
    
strauss.sonify(x, y, style=harp_style)
strauss.display()
strauss.close()

Now let's layer these two styles we've created!

In [ ]:
strauss.sonify(x, y, style=basic_style)
strauss.sonify(x, z, style=harp_style)
strauss.display()
strauss.close()

# Using CSVs and DataFrames

If your data is in a CSV file, or you already have it in a pandas DataFrame, we can also use that in STRAUSS's `sonify()` function.

We can specify the inputs in the Style's `map` section by giving either the column name, or column index (0, 1, 2, etc.).

Let's see an example.

In [ ]:
import pandas as pd

stars_csv = DATA_DIR / 'Stars.csv'

df = pd.read_csv(stars_csv) # Load data into a pandas DataFrame

style = {
    
    "name": "Stars Appearing",
    
    "generator": 
        {
            "type": "sampler",
            "sample": "./samples/Glockenspiel"
        },
    
    "map":
        [
            {
                "input": "magnitude",
                "input_range": ["0%", "120%"], # Adding a bit of time on the end so the last event has time to ring out
                "output": "time"
            },
            {
                "input": "magnitude",
                "function": "invert", # Flip the data so that large magnitude = large volume
                "output": "volume",
                "output_range": [0.1, 1] # Starting at 0.1 so the lowest-volume data point isn't completely silent
            },
            {
                "input": "colour",
                "function": "invert",
                "output": "pitch",      
            },
            {
                "input": "relative_azimuth",
                "output": "pan"
            }
        ],
        
    "sources": "events",
    
    "notes": ["C#3", "F#3", "G#3", "D#4", "F4"]
}

stars_style = str(STYLES_DIR / "stars_appearing.yml")

with open(stars_style, 'w') as file:
    yaml.dump(style, file)
    
strauss.sonify(df, style=stars_style, duration=30)